# Cytomorphology analysis V2 — manuscript figures

One section per display item (main figures, then supplementary). Each figure cell saves the figure to
`figures/manuscript/` (PNG for viewing, PDF for submission) and shows the saved PNG. Write observations in the
template under each figure (legend draft, key observations for Results, caveats).

Plotting code lives in `manuscript_figures/` (one module per figure group); this notebook only calls it.
Exploratory analyses stay in `11_cytomorphology_analysis.ipynb`.

## Inputs

The notebook only reads derived tables. Regenerate them after a new model run (from the repository root):

1. **aml-mil**: run extraction / figure scripts → `aml-mil/results/<RUN>/figures` (Fig 2, Fig 3, calibration)
2. `.venv/bin/python scripts/extract_fig4_fingerprint_results.py` → `derived/fig4_fingerprints/` (Fig 4/5)
3. `.venv/bin/python scripts/extract_error_attention_cache.py` → `derived/error_analysis/sample_attention_summary.parquet`
4. `.venv/bin/python scripts/extract_error_outcome_stats.py` → `derived/error_analysis/morph_outcome_stats.csv`

5. `.venv/bin/python scripts/extract_morphometry_dysplasia_correlation.py` → `derived/morph_dysplasia/` (morphometry of dysplastic cells)
6. `.venv/bin/python scripts/extract_gallery_cells.py` → `derived/gallery/` (top-attention cells of all card targets, clusters + 2D embedding and tiles of Fig 6)
7. `.venv/bin/python scripts/extract_cell_crops.py data/<version>/derived/cell_crops/lists/*.csv` → `derived/cell_crops/` (cell images; needs the image server on
   `localhost:5432` and the NAS). The figure functions write the cell lists: run the notebook once, fetch the crops, run again (cells without an image are drawn as stand-ins).
8. `.venv/bin/python scripts/extract_vaf_by_outcome.py` → `derived/error_analysis/vaf_by_outcome.csv`, `vaf_outcome_stats.csv` (VAF of TP and FN; needs the VAF tables of `config.py`, so run it where they are available; S13)

The Fig 4/5 UMAP (panel A) is computed here from the cohort (`manuscript_figures/umap_centroids.py`).

> **Pixel size:** BM = 1 / 9.034925854 µm/px (`UM_PER_PX` in `manuscript_figures/config.py`; area × MPP² = µm²; checked against cell-type diameters). PB uses the same value, **not yet confirmed** (`UM_PER_PX_UNCONFIRMED`): update `UM_PER_PX["PB"]` once known. Morphometry pixels = BM at the 1232 × 1028 field resolution (2464-px fields are halved) and PB crops. Ratios (N:C, solidity, eccentricity) and Cliff's δ (Fig 4/5) are unaffected.


In [1]:
%load_ext autoreload
%autoreload 2
import io
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from PIL import Image as PILImage

from manuscript_figures import config as C, cards, cohort, fig4, errors, morph_dysplasia, gallery, outcome_pages, vaf

print("aml-mil run:", C.RUN)
print("derived data:", C.DERIVED)
print("output:", C.OUT)


PREVIEW = False  # True: show a downscaled copy of every figure in the notebook (large file); False: link to the saved files


def preview(path, width):
    """Link to the saved figure (PREVIEW = False) or show a downscaled copy (keeps the notebook small; the saved file stays full resolution)."""
    if not PREVIEW:
        rel = Path(path).relative_to(C.REPO).with_suffix("")
        return display(Markdown(f"[{rel.name}.png]({rel}.png) · [pdf]({rel}.pdf)"))
    im = PILImage.open(path); im.thumbnail((int(1.5 * width), 15 * width))
    buf = io.BytesIO(); im.convert("RGB").save(buf, "JPEG", quality=80)
    display(Image(buf.getvalue(), width=width))


def show(fig, name, width=900):
    """Save (PNG + PDF) and show the saved PNG, so the notebook shows exactly the exported file."""
    C.save(fig, name); plt.close(fig)
    preview(C.OUT / f"{name}.png", width)


def external(src_stem, name, width=900):
    """Copy a figure built elsewhere (PNG + PDF if present) into figures/manuscript and show it."""
    if not src_stem.with_suffix(".png").exists():
        return print(f"not built yet: {src_stem}.png")
    C.OUT.mkdir(parents=True, exist_ok=True)
    for ext in (".png", ".pdf"):
        src = src_stem.with_suffix(ext)
        if src.exists():
            shutil.copy(src, C.OUT / f"{name}{ext}")
    preview(C.OUT / f"{name}.png", width)

aml-mil run: AML_V15_OPTIMIZED_PARAMETERS
derived data: /home/mikkop/projects/2-mut-pred/mut-pred-analysis/data/03-06-2026/derived
output: /home/mikkop/projects/2-mut-pred/mut-pred-analysis/figures/manuscript


# Main figures

## Figure 1 — Study design

*Not built yet (placeholder in mockups).*

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-

## Figure 2 — Discovery performance

Built in aml-mil (`fig2_discovery_performance`).

In [2]:
external(C.AML_MIL_FIGS / "fig2_discovery_performance", "Fig2_discovery_performance")

[Fig2_discovery_performance.png](figures/manuscript/Fig2_discovery_performance.png) · [pdf](figures/manuscript/Fig2_discovery_performance.pdf)

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-

## Figure 3 — External validation

Built in aml-mil (`fig3_external_validation`).

In [3]:
external(C.AML_MIL_FIGS / "fig3_external_validation", "Fig3_external_validation")

[Fig3_external_validation.png](figures/manuscript/Fig3_external_validation.png) · [pdf](figures/manuscript/Fig3_external_validation.pdf)

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-

## Figure 4 — Bone marrow: composition, dysplasia, attention and morphometry

In [4]:
show(fig4.make("BM"), "Fig4_BM")

/home/mikkop/projects/2-mut-pred/mut-pred-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/mikkop/projects/2-mut-pred/mut-pred-analysis/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[Fig4_BM.png](figures/manuscript/Fig4_BM.png) · [pdf](figures/manuscript/Fig4_BM.pdf)

**What it shows (legend draft):** A, UMAP of sample-level morphology (features associated with ≥ 1 target as in B, C, E); grey = samples, large dots = centroid of the samples positive for each target. B, cell proportions; C, dysplasia: balloons = Pos vs Neg association (colour = Cliff's δ, size = FDR-adjusted p; shown if FDR < 0.05). D, attention enrichment of each cell type in predicted positives (log); bars = attention spread N_eff/N (median, IQR). E, morphometry (cell area, nucleus area, N:C ratio, nucleus solidity, nucleus eccentricity) per cell type, coloured if FDR < 0.05. C, E: dot = feature further enriched in the model's top-attention cells; grey × = too few top-attention cells / samples to test. Tests require ≥ 10 samples per group.

**Key observations:**
-

**Caveats / open questions:**
- Minimum group size (≥ 10 samples per group, `MIN_SAMPLES` in `extract_fig4_fingerprint_results.py`) is kept. Without it (min 3), BM gains 16 weak associations from 5–9 positive samples, notably megakaryocyte nucleus shape (solidity, eccentricity, compactness) in complex karyotype (n = 9) and −5/del(5q) (n = 8), multinucleated megakaryocytes in complex karyotype (n = 9) and hypolobated megakaryocytes in TP53 (n = 5; δ = 0.69) — biologically plausible (MDS-related dysmegakaryopoiesis); mention as exploratory in text / supplement. The extra tests also push 15 well-powered borderline BM associations (FDR 0.04–0.05) above 0.05; PB gains only n = 3–4 associations.

## Figure 5 — Peripheral blood: composition, dysplasia, attention and morphometry

In [5]:
show(fig4.make("PB"), "Fig5_PB")

/home/mikkop/projects/2-mut-pred/mut-pred-analysis/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[Fig5_PB.png](figures/manuscript/Fig5_PB.png) · [pdf](figures/manuscript/Fig5_PB.pdf)

**What it shows (legend draft):** As Figure 4, PB model. Band and segmented neutrophils share one attention tile (merged class in the PB model).

**Key observations:**
-

**Caveats / open questions:**
-

## Figure 6 — Showcase: cell gallery

Cluster gallery (BM): 8 tiles per target (`manuscript_figures/gallery.py`; tables from `scripts/extract_gallery_cells.py`).
Cell images: `scripts/extract_cell_crops.py`. Pixel size: `UM_PER_PX` in `manuscript_figures/config.py` (BM confirmed).

In [6]:
show(gallery.make(), "Fig6_gallery", width=1000)

[Fig6_gallery.png](figures/manuscript/Fig6_gallery.png) · [pdf](figures/manuscript/Fig6_gallery.pdf)

**Source tables** — one row per tile and one row per cluster (the numbers are not printed on the figure).

In [7]:
import pandas as pd
tiles, clusters = gallery.tables()
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
    display(tiles.round(3))
    display(clusters.round(3))

,target,tile,cell_type,cluster,attention_share,support,n_tp,dominant_label,database_id,album_id,cell_id,sample_probability,sliders,forced
0,PML-RARA,a,Promyelocytes,4,0.182,15,19,Blasts,VisionDB_2021,2286,144,1.000,nucleus_solidity,
1,PML-RARA,b,Promyelocytes,2,0.155,12,19,Blasts,VisionDB_2010,290,5515,1.000,"nucleus_solidity, nucleus_eccentricity",
2,PML-RARA,c,Promyelocytes,0,0.116,12,19,Promyelocytes,VisionDB_2022,2914,91,1.000,nucleus_solidity,
3,PML-RARA,d,Promyelocytes,7,0.105,11,19,Promyelocytes,VisionDB_2016,3250,10448,1.000,nucleus_solidity,
4,PML-RARA,e,Promyelocytes,1,0.097,10,19,Promyelocytes,VisionDB_2010,3658,1593,0.999,"nucleus_solidity, nucleus_eccentricity, nucleu...",
5,PML-RARA,f,Promyelocytes,10,0.092,11,19,Promyelocytes,vimages_new_06.07.21,1312,3381,0.999,"nucleus_solidity, nucleus_eccentricity",
6,PML-RARA,g,Blasts,6,0.087,5,19,Blasts,VisionDB_2021,3418,936,0.998,,
7,PML-RARA,h,Promyelocytes,3,0.054,2,19,Promyelocytes,VisionDB_2016,323,930,0.992,"nucleus_solidity, nucleus_eccentricity, nucleu...",apl_auer
8,CBFB-MYH11,a,Monocytes,8,0.122,5,8,Monocytes,VisionDB_2023,2171,604,0.786,,
9,CBFB-MYH11,b,Blasts,1,0.118,4,8,Blasts,vimages_new_06.07.21,3318,3230,0.792,nucleus_solidity,


,target,cluster,tile,n_cells,attention_share,support,n_tp,dominant_label,dominant_label_share,low_quality_share,old_scanner_share
0,PML-RARA,4,a,1145,0.182,15,19,Blasts,0.404,0.001,0.688
1,PML-RARA,2,b,829,0.155,12,19,Blasts,0.586,0.000,0.852
2,PML-RARA,0,c,1013,0.116,12,19,Promyelocytes,0.994,0.005,0.702
3,PML-RARA,7,d,868,0.105,11,19,Promyelocytes,0.916,0.001,0.622
4,PML-RARA,1,e,786,0.097,10,19,Promyelocytes,0.949,0.006,0.439
5,PML-RARA,10,f,942,0.092,11,19,Promyelocytes,0.915,0.049,0.332
6,PML-RARA,6,g,602,0.087,5,19,Blasts,0.440,0.154,0.404
7,PML-RARA,3,h,845,0.054,2,19,Promyelocytes,0.875,0.005,0.963
8,PML-RARA,11,NaN,708,0.044,3,19,Promyelocytes,0.914,0.021,0.944
9,PML-RARA,5,NaN,613,0.032,1,19,Promyelocytes,0.912,0.051,0.114


**What it shows (legend draft):** Per target, the top-attention cells (attention mass 0.5) of all TP samples (BM model) are clustered on the 384-d MIL cell features (PCA 50 → k-means, k = 12); the 8 largest clusters are shown, one cell each (nearest the cluster centre, from the most confident TP sample not yet shown). Frame = cell-type colour; chips = classifier dysplasia flags. Sliders: only morphometry features associated with the target in the cell's type on which the cell lies beyond the median of the Pos cells of that type (effect direction), ≤ 3 per cell; ▼ = this cell against the middle 50% of Neg (blue) / Pos (orange) cells, zoomed axis (open = outside). Bar = attention share of the clusters (letters = tiles).

**Table columns.** `attention_share`: share of the target's attention in the cluster — each cell's attention divided by its sample's total attention, summed over the cluster's cells, / total (every sample weighs about the same; the 12 clusters sum to 1). `support`: TP samples in which the cluster holds ≥ 5% of that sample's top-cell attention (of `n_tp` TP samples). `forced`: the tile's cell was chosen among the cells of its cluster that carry this classifier flag (PML-RARA: Auer rods; † in the figure). `cell_type`: classifier label of the shown cell; `dominant_label(_share)`: most frequent label in the cluster. `low_quality_share`, `old_scanner_share`: share of the cluster's cells with low-quality score ≥ 0.5 / from 2464-px (older scanner) fields. `sliders`: features shown for the tile.

**Key observations:**
-

**Caveats / open questions:**
- Clusters overlap partially with cell type (NMI 0.28–0.53) and little with scanner (0.03–0.15). CBFB-MYH11: the eosinophil-like cluster (tile e) is supported by 1 of 8 TP samples; top-attention cells per sample are many (≈ 400–550) because attention is flat. TP samples are those of `errors.samples()` (decision threshold of the outcome table). k-means partitions a continuous embedding (see the embedding figure): cluster borders are arbitrary. Clustering checks (6 targets, cluster-cell tables of this notebook): silhouette 0.05–0.11 (no natural clusters); no interior optimum for k (silhouette favours the smallest k = 6, Gaussian-mixture BIC the largest k ≥ 20); PCA(50) vs no PCA vs cosine give similar partitions (ARI ≈ 0.85), attention-weighted k-means, Ward and Gaussian mixture different ones (ARI 0.35–0.57) with similar overlap with cell type (NMI 0.37–0.40) and attention coverage of the 8 largest clusters (0.79–0.85). PML-RARA: Auer-rod-flagged cells are 7.5% of the top-attention cells, in 8 of 19 TP samples, concentrated in cluster 3 (29% of its cells), hence tile h is forced to an Auer-rod cell. Alternatives evaluated (Fig6_gallery_weighted / _medoid, `gallery.make(method)`): attention-weighted k-means changes ~6 of 8 cells without a better representation (mean distance of the cells to the nearest shown cell 11.6 vs 11.5); representative cells (weighted k-medoids) represent the cloud 4% better (11.0) but with lower sample support (e.g. NPM1 down to 5 of 82 samples) and small remote groups; k-means was kept.

## Figure 7 — Clinical impact

*Not built yet.*

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-

# Supplementary figures

| | Item | Source |
|---|---|---|
| S1 | Cohort genetics | here |
| S2 | Cells per sample and performance vs cells per bag (parts a–e) | aml-mil |
| S3 | Tabular baselines vs the MIL model (8 parts) | aml-mil |
| S4 | Calibration | aml-mil |
| S5 | Morphometry of dysplastic cells | here |
| S6 | Embedding of the top-attention cells (Fig 6) | here |
| S7 | Variable cards (13 targets) | here |
| S8 | Top-attention cells of TP and FP samples (13 targets) | here |
| S9 | Errors in context | here |
| S10 | Shift of misclassified samples: composition, dysplasia, QC | here |
| S11–S12 | Shift of misclassified samples: morphometry (BM, PB) | here |
| S13 | VAF of detected and missed positives (BM, PB, FUSION) | here |
| S14 | Variable-specific examples (optional) | here |

Order: cohort and data, model performance and robustness (S2–S4), what the model uses (S5–S8), errors (S9–S14). Each S-number is used in the file names (`S<n>_*` in `figures/manuscript/`); change a number in the heading and the `save` / `external` names of its section together.

## Supplementary Figure S1 — Cohort genetics

Port of notebook 11 "Study cohort characteristics".

In [8]:
show(cohort.make(), "S1_cohort_genetics")

[S1_cohort_genetics.png](figures/manuscript/S1_cohort_genetics.png) · [pdf](figures/manuscript/S1_cohort_genetics.pdf)

**What it shows (legend draft):** A, targets per patient (columns = patients with a BM or PB sample in the modelling cohort, sorted by the most frequent targets); bars = positive / negative / missing, % = positive among patients with a known label. B, co-occurrence: % of patients positive for the row target that are also positive for the column target (denominator = row-positive patients with the column target known).

**Key observations:**
-

**Caveats / open questions:**
- Notebook 11 counted missing labels as negative in the co-occurrence denominator; here they are excluded.

## Supplementary Figure S2 — Cells per sample and performance vs cells per bag

Built in aml-mil (`supp_cells_per_sample`, `supp_max_cells_*`; `extract_cells_per_sample.py`, `extract_max_cells_results.py`). Part a: how many cells a sample has and how many samples each cap of the sweep reduces. Parts b–e: the 1B sweep (3 seeds × 5-fold CV, fixed hyperparameters, CV only). Reference = no cap (the final models; PB: largest cap = all cells). Grey numbers in the Δ tables = mean cells per bag actually used. The sweep needs the 1B runs of MANUSCRIPT_V1; until then parts b–e print "not built yet".

In [9]:
external(C.AML_MIL_FIGS / "supp_cells_per_sample", "S2a_cells_per_sample", width=1000)
for part, name in [("curves_average_precision", "S2b_cells_per_bag_curves_AP"), ("deltas_average_precision", "S2c_cells_per_bag_deltas_AP"),
                   ("curves_roc_auc", "S2d_cells_per_bag_curves_AUROC"), ("deltas_roc_auc", "S2e_cells_per_bag_deltas_AUROC")]:
    external(C.AML_MIL_FIGS / f"supp_max_cells_{part}", name, width=1000)

[S2a_cells_per_sample.png](figures/manuscript/S2a_cells_per_sample.png) · [pdf](figures/manuscript/S2a_cells_per_sample.pdf)

not built yet: /home/mikkop/projects/2-mut-pred/aml-mil/results/AML_V15_OPTIMIZED_PARAMETERS/figures/supp_max_cells_curves_average_precision.png
not built yet: /home/mikkop/projects/2-mut-pred/aml-mil/results/AML_V15_OPTIMIZED_PARAMETERS/figures/supp_max_cells_deltas_average_precision.png
not built yet: /home/mikkop/projects/2-mut-pred/aml-mil/results/AML_V15_OPTIMIZED_PARAMETERS/figures/supp_max_cells_curves_roc_auc.png
not built yet: /home/mikkop/projects/2-mut-pred/aml-mil/results/AML_V15_OPTIMIZED_PARAMETERS/figures/supp_max_cells_deltas_roc_auc.png


**What it shows (legend draft):** a, cells per sample for BM, PB and BM+PB (the sum of both, the unit of the FUSION model; left) and the share of samples with more cells than each cap of the sweep (right). b, d: AP / AUROC against the maximum number of cells per bag, one panel per target, one line per modality, bands = 95% bootstrap CI. c, e: Δ vs no cap per target and cap (bold: paired 95% CI excludes 0).

**Key observations:**
-

**Caveats / open questions:**
- CV only, no external cohorts; hyperparameters fixed across caps. CIs resample patients (one patient list per round scored by every seed and cap); seeds change the folds and are not part of the CI.

## Supplementary Figure S3 — Tabular baselines vs the MIL model

Built in aml-mil (`supp_tabular_*`; `run_tabular_baseline.py` + `extract_performance_results.py`). Elastic-net logistic regression and gradient boosting on sample-level features (D cell differentials, Y dysplasia, M morphometry) against the MIL model (1A), same folds; discovery and external cohorts, AP and AUROC (eight parts).

In [10]:
for model in ("elastic_net", "hist_gb"):
    for metric, m in (("average_precision", "AP"), ("roc_auc", "AUROC")):
        for part in ("discovery", "external"):
            external(C.AML_MIL_FIGS / f"supp_tabular_{model}_vs_1A_{part}_{metric}", f"S3_tabular_{model}_{part}_{m}", width=1000)

[S3_tabular_elastic_net_discovery_AP.png](figures/manuscript/S3_tabular_elastic_net_discovery_AP.png) · [pdf](figures/manuscript/S3_tabular_elastic_net_discovery_AP.pdf)

[S3_tabular_elastic_net_external_AP.png](figures/manuscript/S3_tabular_elastic_net_external_AP.png) · [pdf](figures/manuscript/S3_tabular_elastic_net_external_AP.pdf)

[S3_tabular_elastic_net_discovery_AUROC.png](figures/manuscript/S3_tabular_elastic_net_discovery_AUROC.png) · [pdf](figures/manuscript/S3_tabular_elastic_net_discovery_AUROC.pdf)

[S3_tabular_elastic_net_external_AUROC.png](figures/manuscript/S3_tabular_elastic_net_external_AUROC.png) · [pdf](figures/manuscript/S3_tabular_elastic_net_external_AUROC.pdf)

[S3_tabular_hist_gb_discovery_AP.png](figures/manuscript/S3_tabular_hist_gb_discovery_AP.png) · [pdf](figures/manuscript/S3_tabular_hist_gb_discovery_AP.pdf)

[S3_tabular_hist_gb_external_AP.png](figures/manuscript/S3_tabular_hist_gb_external_AP.png) · [pdf](figures/manuscript/S3_tabular_hist_gb_external_AP.pdf)

[S3_tabular_hist_gb_discovery_AUROC.png](figures/manuscript/S3_tabular_hist_gb_discovery_AUROC.png) · [pdf](figures/manuscript/S3_tabular_hist_gb_discovery_AUROC.pdf)

[S3_tabular_hist_gb_external_AUROC.png](figures/manuscript/S3_tabular_hist_gb_external_AUROC.png) · [pdf](figures/manuscript/S3_tabular_hist_gb_external_AUROC.pdf)

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-

## Supplementary Figure S4 — Calibration

Built in aml-mil (`supp_calibration_*`; `extract_calibration_results.py`): summary of observed/expected and calibration slope per target, then LOESS calibration curves per modality (raw vs cross-fitted logistic recalibration).

In [11]:
external(C.AML_MIL_FIGS / "supp_calibration_summary", "S4_calibration_summary", width=1000)
for mod in ("BM", "PB", "FUSION"):
    external(C.AML_MIL_FIGS / f"supp_calibration_curves_{mod}", f"S4_calibration_{mod}", width=1000)

[S4_calibration_summary.png](figures/manuscript/S4_calibration_summary.png) · [pdf](figures/manuscript/S4_calibration_summary.pdf)

[S4_calibration_BM.png](figures/manuscript/S4_calibration_BM.png) · [pdf](figures/manuscript/S4_calibration_BM.pdf)

[S4_calibration_PB.png](figures/manuscript/S4_calibration_PB.png) · [pdf](figures/manuscript/S4_calibration_PB.pdf)

[S4_calibration_FUSION.png](figures/manuscript/S4_calibration_FUSION.png) · [pdf](figures/manuscript/S4_calibration_FUSION.pdf)

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-

## Supplementary Figure S5 — Morphometry of dysplastic cells

Data: `scripts/extract_morphometry_dysplasia_correlation.py` (redesign of
`figures/mockups/v3/morph_dysplasia_heatmap.py`).

In [12]:
show(morph_dysplasia.make(), "S5_morph_dysplasia")

[S5_morph_dysplasia.png](figures/manuscript/S5_morph_dysplasia.png) · [pdf](figures/manuscript/S5_morph_dysplasia.pdf)

**What it shows (legend draft):** Within-sample comparison of dysplastic (classifier-flagged) and other cells of the same type: colour = median Cliff's δ across samples (red = higher in dysplastic cells), stars = Wilcoxon signed-rank test of the per-sample δs (BH-FDR per modality), shown if |δ| ≥ 0.1; n = samples with ≥ 5 dysplastic and ≥ 5 other cells (rows need ≥ 10). Square = cell lineage.

**Key observations:**
-

**Caveats / open questions:**
- FDR was computed over 7 measures (incl. nucleus and cell compactness, not shown). Cell filters as the feature aggregation (quality-model filtered, low-quality < 0.5, no mitoses).

## Supplementary Figure S6 — Embedding of the top-attention cells (Fig 6)

UMAP of the cells behind Fig 6 (`derived/gallery/cluster_cells.parquet`, `scripts/extract_gallery_cells.py`).

In [13]:
show(gallery.scatter(), "S6_gallery_embedding", width=1000)

[S6_gallery_embedding.png](figures/manuscript/S6_gallery_embedding.png) · [pdf](figures/manuscript/S6_gallery_embedding.pdf)

**What it shows (legend draft):** UMAP (n_neighbors 15, min_dist 0.2) of the 50-d PCA of the 384-d MIL cell features of the top-attention cells (attention mass 0.5) of the TP samples, BM model, per Fig 6 target. Left: coloured by cell type; right: the 8 clusters shown in Fig 6 coloured, other clusters grey. Letters = the tile cells a–h of Fig 6.

**Key observations:**
-

**Caveats / open questions:**
- The embedding is a continuum and k-means cuts it into 12 parts: cluster borders are arbitrary, and UMAP distances are not metric. Tile cells are nearest the cluster centre in the PCA space, so not always central in the UMAP. Cell-type islands (e.g. the immature-eosinophil island of CBFB-MYH11 / −5/del(5q)) separate clearly.

## Supplementary Figure S7 — Variable cards

One page per target in `cards.CARD_VARS` (13 targets, as notebook 11; long cards continue on a second page). B = associated
dysplasia / morphometry features that are model focus (dot in Fig 4/5), one frame per measure × direction (box plots + example
cells with sliders). Add a hand-picked frame (e.g. a single feature that completes the story): `cards.CARDS["TP53"] =
{"extra": [{"panels": [("BM", col), ...]}]}` before the cell below (`"frames"` replaces B).

In [14]:
for v in cards.CARD_VARS:
    for k, f in enumerate(cards.make(v), 1):
        show(f, f"S7_card_{v}_p{k}")

[S7_card_PML-RARA_p1.png](figures/manuscript/S7_card_PML-RARA_p1.png) · [pdf](figures/manuscript/S7_card_PML-RARA_p1.pdf)

[S7_card_CBFB-MYH11_p1.png](figures/manuscript/S7_card_CBFB-MYH11_p1.png) · [pdf](figures/manuscript/S7_card_CBFB-MYH11_p1.pdf)

[S7_card_RUNX1-RUNX1T1_OR_t_8_21_p1.png](figures/manuscript/S7_card_RUNX1-RUNX1T1_OR_t_8_21_p1.png) · [pdf](figures/manuscript/S7_card_RUNX1-RUNX1T1_OR_t_8_21_p1.pdf)

[S7_card_IDH_mutations_p1.png](figures/manuscript/S7_card_IDH_mutations_p1.png) · [pdf](figures/manuscript/S7_card_IDH_mutations_p1.pdf)

[S7_card_CpG_mutations_p1.png](figures/manuscript/S7_card_CpG_mutations_p1.png) · [pdf](figures/manuscript/S7_card_CpG_mutations_p1.pdf)

[S7_card_NPM1_p1.png](figures/manuscript/S7_card_NPM1_p1.png) · [pdf](figures/manuscript/S7_card_NPM1_p1.pdf)

... (4 outputs left out)

[S7_card_complex_karyotype_p1.png](figures/manuscript/S7_card_complex_karyotype_p1.png) · [pdf](figures/manuscript/S7_card_complex_karyotype_p1.pdf)

[S7_card_monosomy5_del_5q_p1.png](figures/manuscript/S7_card_monosomy5_del_5q_p1.png) · [pdf](figures/manuscript/S7_card_monosomy5_del_5q_p1.pdf)

[S7_card_TP53_p1.png](figures/manuscript/S7_card_TP53_p1.png) · [pdf](figures/manuscript/S7_card_TP53_p1.pdf)

[S7_card_STAG2_p1.png](figures/manuscript/S7_card_STAG2_p1.png) · [pdf](figures/manuscript/S7_card_STAG2_p1.pdf)

**What it shows (legend draft):** A, cell differentials (BM, PB): Neg, Pos and the model's top-attention cells of Pos samples; stars = Neg vs Pos FDR, • = enriched in top-attention cells. B, associated dysplasia and morphometric features further enriched in the model's top-attention cells, one frame per measure × direction: box plots (points = samples, coloured by the model's outcome: FP among Neg, FN among Pos; brackets: Neg vs Pos and Pos vs Pos top FDR; 'not tested' = < 10 samples with top-cell values) and example cells of Pos samples (4 typical top-attention cells, TP samples first, + 1 striking cell) with morphometry sliders (▼ = the cell; bars = middle 50% of Neg / Pos cells of its type; areas in µm²).

**Key observations:**
-

**Caveats / open questions:**
- Neg / Pos values from the cohort features (as the association test); Pos top re-aggregated from the top-attention cells (attention-mass threshold 0.5). Typical = closest to the median of the top-attention cells (frame measure counts double); striking = most extreme in the frame measure within the 2nd–98th percentile; cells whose crop reaches over the field edge are skipped. CBFB-MYH11, KIT, TP53 and STAG2 have no model-focus feature (mostly not testable: < 10 samples with top-cell values), so their card shows only A unless frames are added by hand. PB pixel size is assumed equal to BM (not yet confirmed).

## Supplementary Figure S8 — Top-attention cells of true- and false-positive samples

One page per target in `cards.CARD_VARS` (`manuscript_figures/outcome_pages.py`); panels BM TP | BM FP, PB TP | PB FP. Cells chosen by
attention only: random draw from the top-attention cells (attention mass 0.5), ≤ 3 per sample, ≤ 48 per panel, no quality or
cell-type filter; ordered by cell type, then by similarity of the MIL cell features. Numbers on the cells = samples.

In [15]:
for v in cards.CARD_VARS:
    show(outcome_pages.page(v), f"S8_top_cells_{v}")

[S8_top_cells_PML-RARA.png](figures/manuscript/S8_top_cells_PML-RARA.png) · [pdf](figures/manuscript/S8_top_cells_PML-RARA.pdf)

[S8_top_cells_CBFB-MYH11.png](figures/manuscript/S8_top_cells_CBFB-MYH11.png) · [pdf](figures/manuscript/S8_top_cells_CBFB-MYH11.pdf)

[S8_top_cells_RUNX1-RUNX1T1_OR_t_8_21.png](figures/manuscript/S8_top_cells_RUNX1-RUNX1T1_OR_t_8_21.png) · [pdf](figures/manuscript/S8_top_cells_RUNX1-RUNX1T1_OR_t_8_21.pdf)

[S8_top_cells_IDH_mutations.png](figures/manuscript/S8_top_cells_IDH_mutations.png) · [pdf](figures/manuscript/S8_top_cells_IDH_mutations.pdf)

[S8_top_cells_CpG_mutations.png](figures/manuscript/S8_top_cells_CpG_mutations.png) · [pdf](figures/manuscript/S8_top_cells_CpG_mutations.pdf)

[S8_top_cells_NPM1.png](figures/manuscript/S8_top_cells_NPM1.png) · [pdf](figures/manuscript/S8_top_cells_NPM1.pdf)

... (3 outputs left out)

[S8_top_cells_complex_karyotype.png](figures/manuscript/S8_top_cells_complex_karyotype.png) · [pdf](figures/manuscript/S8_top_cells_complex_karyotype.pdf)

[S8_top_cells_monosomy5_del_5q.png](figures/manuscript/S8_top_cells_monosomy5_del_5q.png) · [pdf](figures/manuscript/S8_top_cells_monosomy5_del_5q.pdf)

[S8_top_cells_TP53.png](figures/manuscript/S8_top_cells_TP53.png) · [pdf](figures/manuscript/S8_top_cells_TP53.pdf)

[S8_top_cells_STAG2.png](figures/manuscript/S8_top_cells_STAG2.png) · [pdf](figures/manuscript/S8_top_cells_STAG2.pdf)

**Sample table** — number on the cells → sample, per page and panel (full table saved to `derived/gallery/outcome_pages_samples.csv`); below: samples and cells per panel.

In [16]:
import pandas as pd
st = pd.concat([outcome_pages.samples(v) for v in cards.CARD_VARS])
st.to_csv(C.GALLERY / "outcome_pages_samples.csv", index=False)
with pd.option_context("display.max_rows", None):
    display(st.groupby(["target", "modality", "outcome"]).agg(samples=("number", "size"), cells=("n_cells", "sum")))

samples  cells
target                  modality outcome                
CBFB-MYH11              BM       FP            12     36
                                 TP             8     24
                        PB       FP             4     12
                                 TP             7     21
CpG_mutations           BM       FP            32     48
                                 TP            37     48
                        PB       FP            38     48
                                 TP            42     48
FLT3_mutations          BM       FP            42     48
                                 TP            42     48
                        PB       FP            39     48
                                 TP            43     48
IDH_mutations           BM       FP            21     48
                                 TP            26     48
                        PB       FP            27     48
                                 TP            23     48
KIT                     BM       FP            11     33
                                 TP             2      6
                        PB       FP            16     48
                                 TP             2      6
NPM1                    BM       FP            26     48
                                 TP            34     48
                        PB       FP            29     48
                                 TP            36     48
PML-RARA                BM       FP             5     15
                                 TP            19     48
                        PB       FP             7     21
                                 TP            14     42
RTK_RAS_mutations       BM       FP            36     48
                                 TP            44     48
                        PB       FP            35     48
                                 TP            43     48
RUNX1-RUNX1T1_OR_t_8_21 BM       FP            17     48
                                 TP            14     42
                        PB       FP            16     48
                                 TP            11     33
STAG2                   BM       FP            20     48
                                 TP             6     18
                        PB       FP            19     48
                                 TP             6     18
TP53                    BM       FP            19     48
                                 TP            10     30
                        PB       FP            34     48
                                 TP            11     33
complex_karyotype       BM       FP            26     48
                                 TP            27     48
                        PB       FP            39     48
                                 TP            25     48
monosomy5_del_5q        BM       FP            19     48
                                 TP            22     48
                        PB       FP            23     48
                                 TP            18     48

**What it shows (legend draft):** Top-attention cells (attention-mass threshold 0.5) of true-positive and false-positive samples, BM and PB models, drawn at random (≤ 3 per sample); ordered by cell type, then by similarity of the MIL cell features (TP and FP share one order within a type). Frame and label = cell type; chips = classifier flags of the target's associated dysplasia features; LQ = low-quality score ≥ 0.5; number = sample of the panel (1 = highest predicted probability).

**Key observations:**
-

**Caveats / open questions:**
- Random draw with a fixed seed. Low-quality cells are kept on purpose: the selection shows what the model attends to, without filtering. The numbers show how many samples contribute (e.g. all immature eosinophils in the CBFB-MYH11 BM TP panel come from one sample). Samples whose images cannot be fetched (7 PB samples of VisionDB_2022_veri, 1 BM sample of VisionDB_2017) are skipped (`derived/cell_crops/missing.csv`).

## Supplementary Figure S9 — Errors in context


In [17]:
show(errors.errors_in_context(), "S9_errors_in_context")

[S9_errors_in_context.png](figures/manuscript/S9_errors_in_context.png) · [pdf](figures/manuscript/S9_errors_in_context.pdf)

**What it shows (legend draft):** A, outcome (TP / FN / FP / TN) per sample and target, BM and PB models; bars = counts. B, co-occurrence of false positives (top) and false negatives (bottom) with other abnormalities: Fisher exact log10 OR (Haldane +0.5), solid + counts where FDR < 0.05 (BH within row), faded where p < 0.05; nested target pairs hatched.

**Key observations:**
-

**Caveats / open questions:**
-

## Supplementary Figure S10 — Shift of misclassified samples: composition, dysplasia, QC


In [18]:
show(errors.shift_grid(), "S10_error_shift_grid")

[S10_error_shift_grid.png](figures/manuscript/S10_error_shift_grid.png) · [pdf](figures/manuscript/S10_error_shift_grid.pdf)

**What it shows (legend draft):** Per feature with TP vs TN FDR < 0.05: position of the FP median between TN (0) and TP (1), and of the FN median between TP (0) and TN (1). Red / blue = error group higher / lower than its own class; bold = beyond the other class; cells < 0 hidden; × = < 3 errors; hatched = 3–4 errors. Key at top.

**Key observations:**
-

**Caveats / open questions:**
-

## Supplementary Figures S11–S12 — Shift of misclassified samples: morphometry (BM, PB)


In [19]:
show(errors.shift_grid_morph("BM"), "S11_error_shift_morph_BM")

[S11_error_shift_morph_BM.png](figures/manuscript/S11_error_shift_morph_BM.png) · [pdf](figures/manuscript/S11_error_shift_morph_BM.pdf)

In [20]:
show(errors.shift_grid_morph("PB"), "S12_error_shift_morph_PB")

[S12_error_shift_morph_PB.png](figures/manuscript/S12_error_shift_morph_PB.png) · [pdf](figures/manuscript/S12_error_shift_morph_PB.pdf)

**What it shows (legend draft):** As S10 for morphometric features; rows = features shown in either grid.

**Key observations:**
-

**Caveats / open questions:**
-

## Supplementary Figure S13 — VAF of detected and missed positives

Only mutated patients have a VAF, so the contrast is false negatives (FN) against true positives (TP). One figure per model (BM, PB, FUSION; BM and FUSION use the BM VAF table, PB the PB one).
Data: `derived/error_analysis/vaf_by_outcome.csv`, `vaf_outcome_stats.csv` (`scripts/extract_vaf_by_outcome.py`; needs the VAF tables).


In [21]:
if (C.ERRORS / "vaf_outcome_stats.csv").exists():
    for part, mod in zip("abc", ("BM", "PB", "FUSION")):
        show(vaf.vaf_by_outcome(mod), f"S13{part}_vaf_by_outcome_{mod}")
else:
    print("not built yet: scripts/extract_vaf_by_outcome.py")

not built yet: scripts/extract_vaf_by_outcome.py


**What it shows (legend draft):** VAF (%) of false-negative (FN) and true-positive (TP) samples per target, box = quartiles, points = samples. Above each panel: AUROC of the VAF for TP vs FN (95% bootstrap CI) and the Mann–Whitney p (BH-FDR within model; shown as p < 0.001 below that).

**Key observations:**
-

**Caveats / open questions:**
- VAF is only defined for mutated samples, so false positives cannot be assessed. A low VAF can mean a small subclone, which also lowers the morphological signal; VAF is measured in the sample's own tissue (BM VAF for BM and FUSION).


## Supplementary Figure S14 — Variable-specific examples (optional)

Optional placeholder: use only if the errors need to be shown more granularly than the shift grids (S10–S12). Pick 3–4
targets and take the panels from the shift grids (features where FP / FN samples lie beyond the other class).
Edit `PANELS`: one sub-figure per `(variable, modality, feature)`. Feature = a sample-level column of
`load_cohort(modality)` (e.g. `"Promyelocytes_proportion"`, `"Blasts_nucleus_area_median"`), `"low_quality_all"` /
`"n_cells"`, or a top-attention summary (e.g. `"top_apl_bilobed"`). Current selection is a placeholder.

In [22]:
PANELS = [("PML-RARA", "BM", "Promyelocytes_proportion"),
          ("PML-RARA", "BM", "Myeloid_precursors_hypergranulated_proportion"),
          ("PML-RARA", "PB", "Blasts-apl_hypergranulated_proportion"),
          ("PML-RARA", "BM", "top_apl_bilobed"),
          ("NPM1_AND_FLT3_ITD", "BM", "Lymphocytes_proportion"),
          ("NPM1_AND_FLT3_ITD", "BM", "low_quality_all"),
          ("complex_karyotype", "BM", "Myeloid_precursors_bilobed_proportion"),
          ("CBFB-MYH11", "BM", "Myeloid_precursors_bilobed_proportion")]
show(errors.variable_examples(PANELS), "S14_variable_examples")

[S14_variable_examples.png](figures/manuscript/S14_variable_examples.png) · [pdf](figures/manuscript/S14_variable_examples.pdf)

**What it shows (legend draft):** Four-group distributions (TN, FN, FP, TP); Cliff's δ and unadjusted Mann–Whitney p (illustrative).

**Key observations:**
-

**Caveats / open questions:**
-

# Tables

## Table 1 — Cohort characteristics

*Not built yet.*

**What it shows (legend draft):** 

**Key observations:**
-

**Caveats / open questions:**
-